# Global CAMS Predictions with Aurora Air Pollution

Run Aurora Air Pollution rollouts for a range of initialization dates using
local CAMS NetCDF files from `/data/cams`. No YAML config or fine-tuning
infrastructure required.

In [1]:
# ---- Configuration (edit these) ----
from pathlib import Path

PREDICTION_START = "2024-09-01T00:00:00"   # First initialization time (UTC)
PREDICTION_END   = "2024-09-30T12:00:00"   # Last initialization time (UTC)
ROLLOUT_STEPS    = 6                       # Number of Aurora steps per rollout
AURORA_STEP_HOURS = 12                      # Hours per Aurora step

# Data paths
CAMS_DATA_DIR = Path("/data/cams")
STATIC_PICKLE = CAMS_DATA_DIR / "aurora-0.4-air-pollution-static.pickle"
OUTPUT_DIR    = Path("./outputs/cams_rollouts")

# Model settings
MIXED_PRECISION = "bf16"   # "bf16" or "none"
SAVE_NETCDF = True


In [2]:
import os, re, pickle
import numpy as np
import torch
import xarray as xr
from tqdm.auto import tqdm

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

prediction_start = np.datetime64(PREDICTION_START, "s")
prediction_end = np.datetime64(PREDICTION_END, "s")

# Aurora needs 2 history steps: include one extra step before the first init.
data_start = prediction_start - np.timedelta64(AURORA_STEP_HOURS, "h")


def _parse_cams_file_range(path):
    match = re.search(r"(\d{4}-\d{2}-\d{2})_to_(\d{4}-\d{2}-\d{2})-cams-range-lead0", path.name)
    if not match:
        return None
    start = np.datetime64(match.group(1) + "T00:00:00", "s")
    end = np.datetime64(match.group(2) + "T23:59:59", "s")
    return start, end


def _matching_cams_files(kind):
    matches = []
    for f in sorted(CAMS_DATA_DIR.glob(f"*lead0-{kind}.nc")):
        r = _parse_cams_file_range(f)
        if r and r[1] >= data_start and r[0] <= prediction_end:
            matches.append(f)
    if not matches:
        raise FileNotFoundError(f"No *lead0-{kind}.nc files in {CAMS_DATA_DIR} overlap range")
    return matches


surface_files = _matching_cams_files("surface-level")
atmos_files = _matching_cams_files("atmospheric")

print(f"Surface files: {len(surface_files)}")
print(f"Atmospheric files: {len(atmos_files)}")
print(f"Date range needed: {data_start} -> {prediction_end}")


Surface files: 3
Atmospheric files: 3
Date range needed: 2024-08-31T12:00:00 -> 2024-09-30T12:00:00


/home/azureuser/miniforge3/envs/aurora/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Index CAMS chunks without loading and merging the full date range.

def _normalize_cams_dataset(ds, time_coord="valid_time"):
    # CAMS analysis files have forecast_period=0; squeeze it out.
    if "forecast_period" in ds.dims:
        ds = ds.isel(forecast_period=0)
    # Rename forecast_reference_time -> time for uniform handling.
    if "forecast_reference_time" in ds.dims and "time" not in ds.dims:
        ds = ds.rename({"forecast_reference_time": "time"})
    # valid_time may be a coordinate computed from reference_time + period.
    # Use it as the primary time if it exists and equals time dim.
    if time_coord in ds.coords and "time" in ds.dims:
        ds = ds.assign_coords(time=ds[time_coord].values)
    return ds.sortby("time")


def _time_key(t):
    return str(np.datetime64(t, "s"))


def _time_values_for_file(path):
    with xr.open_dataset(path, engine="netcdf4") as ds:
        ds = _normalize_cams_dataset(ds)
        return ds.time.values.astype("datetime64[s]")


def _build_time_index(files):
    index = []
    for f in files:
        times = _time_values_for_file(f)
        index.append(
            {
                "path": f,
                "start": times[0],
                "end": times[-1],
                "time_keys": {_time_key(t) for t in times},
            }
        )
    return index


def _has_time(index, t):
    key = _time_key(t)
    return any(entry["start"] <= t <= entry["end"] and key in entry["time_keys"] for entry in index)


def _load_times(index, target_times):
    target_times = [np.datetime64(t, "s") for t in target_times]
    remaining = {_time_key(t) for t in target_times}
    chunks = []

    for entry in index:
        selected = [t for t in target_times if _time_key(t) in remaining and _time_key(t) in entry["time_keys"]]
        if not selected:
            continue
        with xr.open_dataset(entry["path"], engine="netcdf4") as ds:
            ds = _normalize_cams_dataset(ds)
            chunks.append(ds.sel(time=selected).load())
        remaining.difference_update(_time_key(t) for t in selected)
        if not remaining:
            break

    if remaining:
        missing = ", ".join(sorted(remaining))
        raise ValueError(f"Missing CAMS timesteps: {missing}")

    merged = xr.concat(chunks, dim="time") if len(chunks) > 1 else chunks[0]
    merged = merged.sortby("time")
    return merged.sel(time=target_times)


def _datetime_range(start, end, step_hours):
    out = []
    t = np.datetime64(start, "s")
    end = np.datetime64(end, "s")
    step = np.timedelta64(step_hours, "h")
    while t <= end:
        out.append(t)
        t = t + step
    return out


surface_index = _build_time_index(surface_files)
atmos_index = _build_time_index(atmos_files)

with xr.open_dataset(atmos_files[0], engine="netcdf4") as sample_atmos:
    sample_atmos = _normalize_cams_dataset(sample_atmos)
    ATMOS_LEVELS = tuple(int(lv) for lv in sample_atmos.pressure_level.values)

candidate_init_times = _datetime_range(prediction_start, prediction_end, AURORA_STEP_HOURS)
init_times = [
    t for t in candidate_init_times
    if _has_time(surface_index, t)
    and _has_time(surface_index, t - np.timedelta64(AURORA_STEP_HOURS, "h"))
    and _has_time(atmos_index, t)
    and _has_time(atmos_index, t - np.timedelta64(AURORA_STEP_HOURS, "h"))
]

print(f"Surface files indexed: {len(surface_index)}")
print(f"Atmospheric files indexed: {len(atmos_index)}")
print(f"Atmos levels: {ATMOS_LEVELS}")
print(f"Valid initialization times: {len(init_times)}")
if init_times:
    print(f"  First: {init_times[0]}")
    print(f"  Last:  {init_times[-1]}")


Surface files indexed: 3
Atmospheric files indexed: 3
Atmos levels: (1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50)
Valid initialization times: 60
  First: 2024-09-01T00:00:00
  Last:  2024-09-30T12:00:00


In [4]:
# Load static variables and define per-initialization batch construction.
import warnings
warnings.filterwarnings("ignore", message=".*non-writable.*")

with open(STATIC_PICKLE, "rb") as f:
    static_vars = pickle.load(f)

from aurora import Batch, Metadata


def build_batch(init_time) -> Batch:
    """Build an Aurora Batch from the two CAMS timesteps ending at init_time."""
    init_time = np.datetime64(init_time, "s")
    history_times = np.array(
        [init_time - np.timedelta64(AURORA_STEP_HOURS, "h"), init_time],
        dtype="datetime64[s]",
    )
    s = _load_times(surface_index, history_times)
    a = _load_times(atmos_index, history_times)

    batch = Batch(
        surf_vars={
            "2t":   torch.from_numpy(s["t2m"].values.copy()[None]),
            "10u":  torch.from_numpy(s["u10"].values.copy()[None]),
            "10v":  torch.from_numpy(s["v10"].values.copy()[None]),
            "msl":  torch.from_numpy(s["msl"].values.copy()[None]),
            "pm1":  torch.from_numpy(s["pm1"].values.copy()[None]),
            "pm2p5": torch.from_numpy(s["pm2p5"].values.copy()[None]),
            "pm10": torch.from_numpy(s["pm10"].values.copy()[None]),
            "tcco": torch.from_numpy(s["tcco"].values.copy()[None]),
            "tc_no": torch.from_numpy(s["tc_no"].values.copy()[None]),
            "tcno2": torch.from_numpy(s["tcno2"].values.copy()[None]),
            "gtco3": torch.from_numpy(s["gtco3"].values.copy()[None]),
            "tcso2": torch.from_numpy(s["tcso2"].values.copy()[None]),
        },
        static_vars={k: torch.from_numpy(np.array(v)) for k, v in static_vars.items()},
        atmos_vars={
            "t":   torch.from_numpy(a["t"].values.copy()[None]),
            "u":   torch.from_numpy(a["u"].values.copy()[None]),
            "v":   torch.from_numpy(a["v"].values.copy()[None]),
            "q":   torch.from_numpy(a["q"].values.copy()[None]),
            "z":   torch.from_numpy(a["z"].values.copy()[None]),
            "co":  torch.from_numpy(a["co"].values.copy()[None]),
            "no":  torch.from_numpy(a["no"].values.copy()[None]),
            "no2": torch.from_numpy(a["no2"].values.copy()[None]),
            "go3": torch.from_numpy(a["go3"].values.copy()[None]),
            "so2": torch.from_numpy(a["so2"].values.copy()[None]),
        },
        metadata=Metadata(
            lat=torch.from_numpy(a.latitude.values.copy()),
            lon=torch.from_numpy(a.longitude.values.copy() % 360),
            time=(history_times[-1].tolist(),),
            atmos_levels=ATMOS_LEVELS,
        ),
    )
    s.close()
    a.close()
    return batch


print(f"Static vars: {list(static_vars.keys())}")
print("Each rollout will load only its two required CAMS history timesteps.")


/tmp/ipykernel_763461/3463314107.py:6: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  static_vars = pickle.load(f)


Static vars: ['lsm', 'z', 'slt', 'static_ammonia', 'static_ammonia_log', 'static_co', 'static_co_log', 'static_nox', 'static_nox_log', 'static_so2', 'static_so2_log']
Each rollout will load only its two required CAMS history timesteps.


In [5]:
# Load model and run one rollout per initialization time.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from aurora import AuroraAirPollution, rollout


def _pick_best_gpu():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    best_idx, best_free = 0, -1
    for i in range(torch.cuda.device_count()):
        torch.cuda.set_device(i)
        free, _ = torch.cuda.mem_get_info()
        if free > best_free:
            best_idx, best_free = i, free
    print(f"Using GPU {best_idx} ({best_free / 1024**3:.1f} GiB free)")
    return torch.device(f"cuda:{best_idx}")


device = _pick_best_gpu()

# Model stays fp32 but uses internal autocast for bf16 backbone (saves memory).
model = AuroraAirPollution(autocast=True)
model.load_checkpoint("microsoft/aurora", "aurora-0.4-air-pollution.ckpt", strict=False)
model.eval()
model = model.to(device)
print(f"Model loaded on {device}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

total_steps = len(init_times) * ROLLOUT_STEPS
rollout_results = []
pbar = tqdm(total=total_steps, desc="Aurora rollout", unit="step")

for init_time in init_times:
    pbar.set_postfix(init=str(init_time), refresh=False)

    batch = build_batch(init_time)

    ds_vars = {}
    lead_times = []
    lat = None
    lon = None

    with torch.inference_mode():
        for step_i, pred in enumerate(rollout(model, batch, steps=ROLLOUT_STEPS), 1):
            lead_times.append(step_i * AURORA_STEP_HOURS)
            if lat is None:
                lat = pred.metadata.lat.cpu().numpy()
                lon = pred.metadata.lon.cpu().numpy()
            # Move to CPU immediately to free GPU memory for next step.
            for vname, tensor in pred.surf_vars.items():
                ds_vars.setdefault(vname, []).append(
                    tensor[0, -1].cpu().float().numpy()
                )
            for vname, tensor in pred.atmos_vars.items():
                ds_vars.setdefault(vname, []).append(
                    tensor[0, -1].cpu().float().numpy()
                )
            del pred
            torch.cuda.empty_cache()
            pbar.update(1)

    del batch
    torch.cuda.empty_cache()

    if SAVE_NETCDF and ds_vars:
        init_tag = str(init_time).replace("-", "").replace(":", "").replace("T", "_")
        out_path = OUTPUT_DIR / f"rollout_{init_tag}.nc"

        valid_times = np.array(
            [init_time + np.timedelta64(int(hours), "h") for hours in lead_times],
            dtype="datetime64[s]",
        )

        coords = {
            "time": valid_times,
            "lead_time": ("time", np.array(lead_times)),
            "latitude": lat,
            "longitude": lon,
            "level": np.array(ATMOS_LEVELS),
        }
        data_vars = {}
        for vname, arrays in ds_vars.items():
            stacked = np.stack(arrays, axis=0)
            if stacked.ndim == 3:  # (time, H, W) - surface
                data_vars[vname] = (["time", "latitude", "longitude"], stacked)
            else:  # (time, L, H, W) - atmospheric
                data_vars[vname] = (["time", "level", "latitude", "longitude"], stacked)

        out_ds = xr.Dataset(data_vars, coords=coords)
        out_ds.attrs["initialization_time"] = str(init_time)
        out_ds.attrs["step_hours"] = AURORA_STEP_HOURS
        out_ds.to_netcdf(str(out_path))
        out_ds.close()
        rollout_results.append({"init_time": init_time, "path": out_path})

pbar.close()
print(f"Done: {len(rollout_results)} rollouts saved to {OUTPUT_DIR}")

# Free GPU memory so other notebooks can use it.
del model
torch.cuda.empty_cache()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
print("GPU memory released.")


Using GPU 0 (90.7 GiB free)
Model loaded on cuda:0


Aurora rollout: 100%|██████████| 360/360 [16:37<00:00,  2.77s/step, init=2024-09-30T12:00:00]

Done: 60 rollouts saved to outputs/cams_rollouts
GPU memory released.


In [6]:
# Summary of saved rollouts.
for r in rollout_results:
    print(f"  {r['init_time']}  ->  {r['path'].name}")


  2024-09-01T00:00:00  ->  rollout_20240901_000000.nc
  2024-09-01T12:00:00  ->  rollout_20240901_120000.nc
  2024-09-02T00:00:00  ->  rollout_20240902_000000.nc
  2024-09-02T12:00:00  ->  rollout_20240902_120000.nc
  2024-09-03T00:00:00  ->  rollout_20240903_000000.nc
  2024-09-03T12:00:00  ->  rollout_20240903_120000.nc
  2024-09-04T00:00:00  ->  rollout_20240904_000000.nc
  2024-09-04T12:00:00  ->  rollout_20240904_120000.nc
  2024-09-05T00:00:00  ->  rollout_20240905_000000.nc
  2024-09-05T12:00:00  ->  rollout_20240905_120000.nc
  2024-09-06T00:00:00  ->  rollout_20240906_000000.nc
  2024-09-06T12:00:00  ->  rollout_20240906_120000.nc
  2024-09-07T00:00:00  ->  rollout_20240907_000000.nc
  2024-09-07T12:00:00  ->  rollout_20240907_120000.nc
  2024-09-08T00:00:00  ->  rollout_20240908_000000.nc
  2024-09-08T12:00:00  ->  rollout_20240908_120000.nc
  2024-09-09T00:00:00  ->  rollout_20240909_000000.nc
  2024-09-09T12:00:00  ->  rollout_20240909_120000.nc
  2024-09-10T00:00:00  ->  r